Symbolic AI Concepts (Part 2)
## Symbolic Reasoning Foundations for Neural Researchers

This notebook is for people who are strong in **deep learning** but want the minimum set of **symbolic reasoning** skills needed for Neuro‑Symbolic AI.

### What this notebook covers
- Logic programming intuition (Prolog/Datalog concepts)
- Forward vs backward chaining

In [ ]:
from itertools import product
from dataclasses import dataclass

def hr(title: str):
    print("\n" + "="*80)
    print(title)
    print("="*80)

## 1) Logic Programming Intuition (Prolog / Datalog)

Logic programming feels different from Python:
- you *declare* facts and rules
- then you ask a query like: “Who is impacted?” or “Which resources violate a policy?”
- the engine tries to *prove* answers using **unification** and **search/backtracking**

We won’t implement Prolog, but we can mimic query‑style programming: return variable bindings that satisfy rule patterns.

Example query:
> Find all services `A` that depend (directly or indirectly) on a down service.


In [ ]:
hr("Query-style results: all impacted services")

DependsOn = {
    ("frontend", "payments"),
    ("frontend", "search"),
    ("payments", "db"),
    ("search", "db"),
}
Down = {"db"}

# We'll compute transitive closure (reachability) and then answer queries.
def transitive_depends(dep_edges):
    reachable = set(dep_edges)
    changed = True
    while changed:
        changed = False
        for (a, b) in list(reachable):
            for (b2, c) in list(reachable):
                if b == b2 and (a, c) not in reachable:
                    reachable.add((a, c))
                    changed = True
    return reachable

Reach = transitive_depends(DependsOn)

def query_impacted():
    for (a, b) in Reach:
        if b in Down:
            yield a, b  # (service, down_dependency)

print("Answers to query Impacted(A) because Down(B):")
for a, b in sorted(query_impacted()):
    print(f"- {a} (depends on {b})")


## 2) Forward vs Backward Chaining

Two ways to do inference:

### Forward chaining (data‑driven)
- Start from known facts
- Repeatedly apply rules to derive new facts
- Good when you want **all consequences** precomputed (materialization)

In [ ]:
hr("Forward chaining: compute all reachable pairs")

Edge = {("A","B"), ("B","C"), ("C","D"), ("A","E")}

Reachable = set(Edge)  # base rule: Edge -> Reachable

changed = True
while changed:
    changed = False
    for (x, y) in list(Reachable):
        for (y2, z) in list(Reachable):
            if y == y2 and (x, z) not in Reachable:
                Reachable.add((x, z))  # transitive rule
                changed = True

print("Reachable pairs:")
for p in sorted(Reachable):
    print(" ", p)

### Backward chaining (goal‑driven)
- Start from a goal/query (e.g., `Impacted(frontend)?`)
- Work backwards: “what must be true for this to hold?”
- Good when you want to answer **specific queries** without computing everything

In [ ]:
hr("Backward chaining: prove Reachable(A, D) by goal-directed search")

def prove_reachable(start, target, seen=None):
    # Think: try to prove the goal by exploring facts that could make it true.
    if seen is None:
        seen = set()
    if start == target:
        return True
    if start in seen:
        return False
    seen.add(start)

    # Backward/goal-directed: to prove Reachable(start, target),
    # try to find an outgoing edge start->next and prove Reachable(next, target).
    for (u, v) in Edge:
        if u == start:
            if prove_reachable(v, target, seen):
                return True
    return False

print("Goal Reachable(E, D)?", prove_reachable("E", "D"))
